# Longitudinal tracking simulations


Current tutors: S. Albright, H. Damerau, A. Lasheen, M. Taquet

Contributors: J. Flowerdew, L. Intelisano, D. Quartullo, F. Tecker, M. Zampetakis



## Links


- Introductory CAS website: https://indico.cern.ch/event/1622828/
- Programme of the CAS: https://indico.cern.ch/event/1622828/attachments/3196386/5990113/Timetable_Introductory2026_ver1.6b.pdf
- Python software installation for longitudinal exercises: https://github.com/cerncas/hands-on-longitudinal-exercises/blob/main/README.md
- Longitudinal hands-on, link to content and cheat sheets: https://indico.cern.ch/event/1622828/contributions/7202999/


## Introduction

In this hands-on session we will experiment with particle tracking simulations.

The goal of the session is to write a tracking code to observe the evolution of the particles in the longitudinal phase space ($\phi$, $\Delta E$), for each turn in the machine.

The notebook is constructed with the following purpose in mind:

1. Compute basic machine parameters (the example of the CERN scSPS is taken).
2. Write the equations of motion in the form of python functions, to track a single particle.
3. Record and observe the trajectory of a particle in the longitudinal phase space.
4. Extend the tracking code to work with many particles, acceleration, below and above transition.
5. Optionally, analyze the particle motion and compare with analytical evaluations of the bucket area, height, synchrotron frequency.

In the code cells, the two tracking functions to complete are marked with `# FILL`. The other cells only give you the first variables and a few hints as comments: the rest is yours, and you are encouraged to experiment.

**Would you rather not write code?** The companion notebook `LongitudinalHandsOnTrackingAnimations` contains a complete tracking code. There you only change input parameters and observe the motion of a bunch of particles with animations, e.g. the injection of a bunch that is matched or not to the RF bucket.

Along the exercises, you will be encouraged to use `support_functions`. These were designed to help you during the hands-on session by reducing the coding overhead. You can check the documentation of each function by calling `function?` in a new cell.

The available support functions are

| `support_functions.py`        | Purpose                                                   |
| ----------------------------- | --------------------------------------------------------- |
| plot_phase_space_trajectory   | Plot the trajectories of a few particles                  |
| separatrix                    | Compute the separatrix to draw on top of the plots        |
| oscillation_spectrum          | Spectrum of the oscillation of a particle                 |
| synchrotron_tune              | Synchrotron tune of a particle, from its spectrum         |
| generate_bunch                | Generate a bunch of particles (animations notebook)       |
| plot_phase_space_distribution | Plot a bunch of particles with its profiles (animations notebook) |
| run_animation                 | Animate the tracking of a bunch of particles (animations notebook) |

The *Lecture slides* lines below point to the slides of the **Longitudinal Dynamics** lecture (F. Tecker) and of the **RF Systems** lecture (C. Völlinger) at this school, where the corresponding topics are introduced. The numbers are the slide numbers printed at the bottom of the slides.

## Importing modules


In [ ]:
# In this cell we import all modules that will be required for the computation
# You can add extra imports as you progress in the exercises
# Hint: use scipy.constants for elementary charge, speed of light, mass of a proton...

import matplotlib.pyplot as plt
import numpy as np
from scipy.constants import c, e, m_p

## Basic accelerator and beam parameters


### Parameters of the superconducting Super Proton Synchrotron (scSPS) at CERN

| Parameter                        |                                                                                                                                |
| -------------------------------- | ------------------------------------------------------------------------------------------------------------------------------ |
| Energy range                     | $E_\mathrm{kin} = 26\,\mathrm{GeV}...1300\,\mathrm{GeV}$                                                                       |
| Circumference                    | $2 \pi R = 6911.5\,\mathrm{m}$                                                                                                 |
| Bending radius                   | $\rho = 741.3\,\mathrm{m}$                                                                                                     |
| Transition gamma                 | $\gamma_\mathrm{tr} = 18.$                                                                                                     |
| Acceleration time                | $4\,\mathrm{s}$                                                                                                                |
| Harmonic number                  | $4620$                                                                                                                         |
| RF voltage at injection          | $V_{\mathrm{rf,inj}} = 4.5\,\mathrm{MV}$                                                                                            |
| Maximum RF voltage               | $V_{\mathrm{rf,max}} = 15\,\mathrm{MV}$                                                                                            |
| Longitudinal emittance per bunch | $\varepsilon_\mathrm{l} = 0.6\,\mathrm{eVs}$                                                                                   |
| Maximum bucket filling factor    | $\varepsilon_\mathrm{l}/A_\mathrm{bucket} = 0.8$                                                                               |
| Total beam intensity             | $N = 1.6 \cdot 10^{14} \,\mathrm{protons}$ $(2 \times 320\mathrm{b} \times 2.5 \cdot 10^{11} \mathrm{protons}/\mathrm{bunch})$ |


## Exercise 1: Compute basic machine parameters

*Lecture slides: Tecker 4, 15, 27–29, 32*


1. Compute the following parameters at the minimum/maximum energies
   -  $E$, $p$
   -  $\beta$, $\gamma$, $T_{\mathrm{rev}}$, $f_{\mathrm{rev}}$
   -  $f_{\mathrm{rf}}$, $T_{\mathrm{rf}}$
   -  $\alpha_c$, $\eta$
   - *The computation is given below: run it at the minimum and at the maximum energy*
2. Some reflection and crosscheck with respect to yesterday's hands-on exercises
   - How large is the RF frequency sweep ?
   - What is the bucket length ?
   - Are we above/below transition ? *(Tecker 29-32)*

In [ ]:
# Machine and beam parameters at injection, used for the tracking in the following exercises

Ekin = 26e9  # eV
voltage = 4.5e6  # V

charge = 1  # in units of e
circumference = 6911.5  # m
harmonic = 4620
gamma_t = 18

# Energy and momentum, in eV and eV/c
E0 = m_p * c**2 / e
energy = Ekin + E0
momentum = np.sqrt(energy**2 - E0**2)
beta = momentum / energy
gamma = energy / E0

# Revolution and rf frequencies
t_rev = circumference / (beta * c)
f_rev = 1 / t_rev
f_rf = harmonic * f_rev
t_rf = 1 / f_rf

# Momentum compaction and phase slippage factor
alpha_c = 1 / gamma_t**2
eta = alpha_c - 1 / gamma**2

print("Kinetic energy: " + str(Ekin / 1e9) + " GeV")
print("Beta: " + str(beta))
print("Gamma: " + str(gamma))
print("Revolution period: " + str(t_rev * 1e6) + " mus")
print("RF frequency: " + str(f_rf / 1e6) + " MHz")
print("RF period: " + str(t_rf * 1e9) + " ns")
print("Momentum compaction factor: " + str(alpha_c))
print("Phase slippage factor: " + str(eta))

In [ ]:
# Questions of point 2, calculator style, with the values printed above at both energies

# --- Solution ---
# RF frequency sweep in kHz: RF frequency at 1.3 TeV minus RF frequency at 26 GeV, in kHz
print((200.396556 - 200.275014) * 1e3, "kHz")

# Bucket length in ns: one RF period
print(1 / 200.275014e6 * 1e9, "ns")

# Transition: gamma is 28.7 at 26 GeV and 1386.5 at 1.3 TeV, always larger than gamma_t = 18,
# and eta is positive: we are above transition during the whole cycle

## Exercise 2: Tracking with a single particle

*Lecture slides: Tecker 19, 34, 45–47, 58*


1. Write functions to track the particle coordinates following longitudinal equations of motion *(Tecker 46–47, 58)*
   $$\phi_{n+1} = \phi_n + 2 \pi h \eta \frac{\Delta E_n}{\beta^2 E}$$
   $$\Delta E_{n+1} = \Delta E_n + q V \sin (\phi_{n+1}) - U_0$$
   - *Start with no acceleration ($U_0 = 0$) or synchrotron radiation*
2. Define the initial coordinates of a particle in the $(\phi, \Delta E)$ phase space.
   - *Hint: maybe avoid 0 and $\pi$ for the phase...*
3. Simulate the evolution of the particle coordinates for a few hundred turns and store the coordinates.
   - *Pre-allocate numpy arrays with `np.zeros(n_turns)` and fill in the particle coordinates each turn*
4. Plot the evolution of the particle phase and energy vs. turn number, and the particle motion in longitudinal phase space *(Tecker 34, 37)*
   - *The support function `plot_phase_space_trajectory` can be used*


In [ ]:
# Tracking functions, one turn each
# The drift comes first: the phase is updated with the energy of the previous turn,
# then the kick uses the new phase (see the indices n and n+1 in the equations)


def drift(phaseInitial, energyInitial, harmonic, eta, beta, energy):
    newPhase = phaseInitial + 2 * np.pi * harmonic * eta * energyInitial / (
        beta**2 * energy
    )  # FILL
    return newPhase


def kick(energyInitial, phaseInitial, charge, voltage, acceleration=0):
    newEnergy = energyInitial + charge * voltage * np.sin(phaseInitial) - acceleration  # FILL
    return newEnergy

In [ ]:
# Initial coordinates of the particle
# - choose a phase (in rad) and an energy offset (in eV) for your particle

# --- Solution ---
particlePhase = 0.99 * np.pi
particleEnergy = 0

In [ ]:
# Tracking, storing the coordinates at each turn

n_turns = 1000

# - pre-allocate two arrays with np.zeros(n_turns), for the phase and for the energy
# - loop over the turns with: for i in range(n_turns):
# - at each turn store the coordinates in the arrays, then apply your drift and your kick

# --- Solution ---
particlePhaseArray = np.zeros(n_turns)
particleEnergyArray = np.zeros(n_turns)
for i in range(n_turns):
    particlePhaseArray[i] = particlePhase
    particleEnergyArray[i] = particleEnergy
    particlePhase = drift(particlePhase, particleEnergy, harmonic, eta, beta, energy)
    particleEnergy = kick(particleEnergy, particlePhase, charge, voltage)

In [ ]:
# Evolution of the coordinates
# - plot the phase vs. turn number, the energy vs. turn number, and the energy vs. phase
# - plt.figure() opens a new figure, plt.plot(x, y) or plt.plot(y) draws a curve,
#   plt.xlabel("...") and plt.ylabel("...") name the axes

# --- Solution ---
plt.figure()
plt.plot(particlePhaseArray)
plt.xlabel("Turn")
plt.ylabel("Phase [rad]")

plt.figure()
plt.plot(particleEnergyArray / 1e6)
plt.xlabel("Turn")
plt.ylabel("Energy [MeV]")

plt.figure()
plt.plot(particlePhaseArray, particleEnergyArray / 1e6)
plt.xlabel("Phase [rad]")
plt.ylabel("Energy [MeV]")

In [ ]:
from support_functions import plot_phase_space_trajectory

plot_phase_space_trajectory?

In [ ]:
# Same plot with the support function
# - call plot_phase_space_trajectory with your two arrays

# --- Solution ---
plot_phase_space_trajectory(particlePhaseArray, particleEnergyArray)

## Exercise 3: Track with a few particles at different amplitudes

*Lecture slides: Tecker 38–40, 52, 83*


1. Repeat the same operations as in Exercise 2, starting with several particles
    - *A numpy array can represent several particles, e.g. `np.array([phase_1, phase_2, phase_3])`, and your `drift` and `kick` functions work on arrays without any change*
    - *Turn your tracking loop into a function `track(...)` returning the trajectories as arrays of size `(n_turns, n_particles)`, so that you do not have to rewrite the loop for every exercise*
2. Track the evolution of particles including with large offsets in $\phi$ and $\Delta E$
    - *What happens when particles are too far from the synchronous particle?* *(Tecker 40)*
3. Generate about 10 particles at $\Delta E=0$ with different phases, and simulate for a few turns
    - *You can use `np.linspace(phase_start, phase_end, n_particles)` to linearly space particles in phase*
    - *What can you observe regarding the velocity of the particles in phase space, vs. the maximum amplitude in phase?* *(Tecker 39)*
4. Plot the separatrix on top of your plot *(Tecker 38, 52)*
    - *Use the `separatrix` support_function to generate the separatrix.*
    - *You can pass the separatrix to the `plot_phase_space_trajectory` to combine plots*


In [ ]:
# Tracking several particles: two ways, choose the one you prefer
# - copy your tracking loop of Exercise 2 in the cells below and adapt it to several particles:
#   the arrays storing the trajectories become np.zeros((n_turns, n_particles))
# - or, if you feel like it, write the loop once as a function track(...) that takes the initial
#   coordinates, the number of turns and the machine parameters, and returns the two trajectories

# --- Solution ---
# The tracking loop of Exercise 2 as a function, for any number of particles
# particlePhase and particleEnergy can be numbers (one particle) or arrays (several particles)


def track(
    particlePhase,
    particleEnergy,
    n_turns,
    harmonic,
    eta,
    beta,
    energy,
    charge,
    voltage,
    acceleration=0,
):
    n_particles = len(particlePhase)
    particlePhaseArray = np.zeros((n_turns, n_particles))
    particleEnergyArray = np.zeros((n_turns, n_particles))
    for i in range(n_turns):
        particlePhaseArray[i] = particlePhase
        particleEnergyArray[i] = particleEnergy
        particlePhase = drift(particlePhase, particleEnergy, harmonic, eta, beta, energy)
        particleEnergy = kick(particleEnergy, particlePhase, charge, voltage, acceleration)
    return particlePhaseArray, particleEnergyArray

In [ ]:
# A few particles at different phases

n_turns = 1000

# - define the initial phases and energies of 3 particles as arrays, e.g. np.array([..., ..., ...])
# - track them, with your loop or with your track function
# - plot the trajectories with plot_phase_space_trajectory

# --- Solution ---
particlePhase = np.array([0.9 * np.pi, np.pi / 2, np.pi / 8])
particleEnergy = np.array([0, 0, 0])

particlePhaseArray, particleEnergyArray = track(
    particlePhase, particleEnergy, n_turns, harmonic, eta, beta, energy, charge, voltage
)

plot_phase_space_trajectory(particlePhaseArray, particleEnergyArray)

In [ ]:
# About ten particles at different phases, with the separatrix

from support_functions import separatrix

n_particles = 10
n_turns = 200

# - space the particles linearly in phase with np.linspace, all at zero energy offset, and track them
# - compute the separatrix with the separatrix function (separatrix? shows its arguments),
#   on an array of phases covering the bucket, e.g. from 0 to 2 pi
# - pass phase_sep and separatrix_array to plot_phase_space_trajectory to draw it on top
# - then try particles at different energies, or much further away in phase

# --- Solution ---
particlePhase = np.linspace(0, np.pi, n_particles)
particleEnergy = np.zeros(n_particles)

particlePhaseArray, particleEnergyArray = track(
    particlePhase, particleEnergy, n_turns, harmonic, eta, beta, energy, charge, voltage
)

phase_array = np.linspace(0, 2 * np.pi, 1000)
phase_sep, separatrix_array = separatrix(
    phase_array, f_rev, eta, beta, energy, charge, voltage, harmonic
)

plot_phase_space_trajectory(
    particlePhaseArray, particleEnergyArray, phase_sep=phase_sep, separatrix_array=separatrix_array
)

# Answer: the larger the amplitude, the slower the particle turns around the bucket,
# the motion stalls on the separatrix (non-linear synchrotron motion, see Exercise 7)

In [ ]:
# --- Solution ---
# Ten particles at different energies

particlePhase = np.ones(n_particles) * np.pi
particleEnergy = np.linspace(0, 2e8, n_particles)

particlePhaseArray, particleEnergyArray = track(
    particlePhase, particleEnergy, n_turns, harmonic, eta, beta, energy, charge, voltage
)

plot_phase_space_trajectory(
    particlePhaseArray, particleEnergyArray, phase_sep=phase_sep, separatrix_array=separatrix_array
)

# Answer: particles outside the separatrix are not captured, their phase is not bounded
# and they drift away from the bucket

In [ ]:
# --- Solution ---
# Particles far away in phase

n_turns = 2000

particlePhase = np.linspace(0, 15, n_particles)
particleEnergy = np.zeros(n_particles)

particlePhaseArray, particleEnergyArray = track(
    particlePhase, particleEnergy, n_turns, harmonic, eta, beta, energy, charge, voltage
)

phase_array = np.linspace(0, 20, 10000)
phase_sep, separatrix_array = separatrix(
    phase_array, f_rev, eta, beta, energy, charge, voltage, harmonic
)

plot_phase_space_trajectory(
    particlePhaseArray, particleEnergyArray, phase_sep=phase_sep, separatrix_array=separatrix_array
)

# Answer: they end up in the neighbouring buckets

## Exercise 4: Acceleration

*Lecture slides: Tecker 20, 31, 42–43*


1. Track the particles by adding the acceleration term, which can be evaluated from the parameter table above *(Tecker 20)*
    - *The energy gain per turn $U_0$ is the total energy gain divided by the number of turns during the ramp*
    - *For simplicity we will neglect here the variations in $\beta$, $\gamma$, $T$, $\omega$...*
2. What is the influence on the particle trajectories ? *(Tecker 42–43)*
    - *You can repeat the same tests as in the previous exercises.*
    - *The `separatrix` function has an `acceleration` argument*
3. What happens if the ramp rate is twice as fast? Or the voltage is halved?
4. What happens if the beam is decelerated instead?


In [ ]:
# Energy gain per turn, from the ramp of the parameter table

Ekin_top = 1.3e12  # eV
Ekin_bottom = 26e9  # eV
t_ramp = 4  # s
voltage = 15e6  # V

# - compute the number of turns during the ramp, from t_ramp and the revolution frequency f_rev
# - the energy gain per turn U0 is the total energy gain divided by that number of turns
# - the stable phase follows from U0 = q V sin(phi_s)

# --- Solution ---
n_turns_ramp = t_ramp * f_rev
U0 = (Ekin_top - Ekin_bottom) / n_turns_ramp

print("Energy gain per turn: " + str(U0 / 1e6) + " MeV")
print("Stable phase: " + str(np.arcsin(U0 / (charge * voltage))) + " rad")

In [ ]:
# Same tracking as in Exercise 3, with acceleration

n_particles = 10
n_turns = 1000

# - track the particles again, with the acceleration U0 in the kick
# - the separatrix function also has an acceleration argument
# - plot_phase_space_trajectory has xlim and ylim arguments (in rad and MeV) if you need to zoom
# - for questions 3 and 4, change U0 or the voltage and run again

# --- Solution ---
particlePhase = np.linspace(0, np.pi, n_particles)
particleEnergy = np.zeros(n_particles)

particlePhaseArray, particleEnergyArray = track(
    particlePhase,
    particleEnergy,
    n_turns,
    harmonic,
    eta,
    beta,
    energy,
    charge,
    voltage,
    acceleration=U0,
)

phase_array = np.linspace(0, 2 * np.pi, 1000)
phase_sep, separatrix_array = separatrix(
    phase_array, f_rev, eta, beta, energy, charge, voltage, harmonic, acceleration=U0
)

plot_phase_space_trajectory(
    particlePhaseArray,
    particleEnergyArray,
    phase_sep=phase_sep,
    separatrix_array=separatrix_array,
    xlim=(-np.pi, 2 * np.pi),
    ylim=(-200, 200),
)

# Answer: the bucket centre moves to the stable phase, the bucket shrinks and becomes asymmetric,
# particles outside are not accelerated and lose energy with respect to the synchronous one
# - ramp twice as fast, or voltage halved: U0/(qV) doubles from 0.49 to 0.98 and the bucket
#   almost vanishes (above 1 there is no stable phase and no bucket at all)
# - deceleration: U0 < 0, the stable phase moves to the other side of pi, the bucket is mirrored

## Exercise 5: On the other side of transition energy

*Lecture slides: Tecker 31–33, 50*


1. What if the injection kinetic energy was 14 GeV instead of 26 GeV?
2. What does that change for the bucket and why?


In [ ]:
# Parameters at 14 GeV, no acceleration
# - copy the cell of Exercise 1 here, with Ekin = 14e9 and voltage = 15e6, to overwrite all the values

# --- Solution ---

Ekin = 14e9  # eV
voltage = 15e6  # V

charge = 1  # in units of e
circumference = 6911.5  # m
harmonic = 4620
gamma_t = 18

# Energy and momentum, in eV and eV/c
E0 = m_p * c**2 / e
energy = Ekin + E0
momentum = np.sqrt(energy**2 - E0**2)
beta = momentum / energy
gamma = energy / E0

# Revolution and rf frequencies
t_rev = circumference / (beta * c)
f_rev = 1 / t_rev
f_rf = harmonic * f_rev
t_rf = 1 / f_rf

# Momentum compaction and phase slippage factor
alpha_c = 1 / gamma_t**2
eta = alpha_c - 1 / gamma**2

print("Kinetic energy: " + str(Ekin / 1e9) + " GeV")
print("Beta: " + str(beta))
print("Gamma: " + str(gamma))
print("Revolution period: " + str(t_rev * 1e6) + " mus")
print("RF frequency: " + str(f_rf / 1e6) + " MHz")
print("RF period: " + str(t_rf * 1e9) + " ns")
print("Momentum compaction factor: " + str(alpha_c))
print("Phase slippage factor: " + str(eta))

In [ ]:
# Tracking below transition

n_particles = 10
n_turns = 1000

# - track about ten particles as in Exercise 3, and plot them with the separatrix
# - where is the bucket now? You may need another range of phases for the separatrix

# --- Solution ---
particlePhase = np.linspace(0, np.pi, n_particles)
particleEnergy = np.zeros(n_particles)

particlePhaseArray, particleEnergyArray = track(
    particlePhase, particleEnergy, n_turns, harmonic, eta, beta, energy, charge, voltage
)

phase_array = np.linspace(-np.pi, np.pi, 1000)
phase_sep, separatrix_array = separatrix(
    phase_array, f_rev, eta, beta, energy, charge, voltage, harmonic
)

plot_phase_space_trajectory(
    particlePhaseArray, particleEnergyArray, phase_sep=phase_sep, separatrix_array=separatrix_array
)

# Answer: below transition (gamma < gamma_t), eta < 0, the stable phase is 0 instead of pi
# and the rotation in phase space is reversed. As |eta| is smaller than at 26 GeV,
# the bucket is also higher and the synchrotron frequency lower

## Exercise 6 (optional): Comparison with analytical evaluations

*Lecture slides: Tecker 49–50, 53, 56*

1. Calculate the height and area of the stationary bucket analytically for lower/higher beam energies. *(Tecker 53, 56)*
2. Determine analytically the synchrotron frequency at the centre of the stationary bucket. *(Tecker 49–50)*
3. Compare qualitatively the results you obtained with the tracking
   - *The formulas are in the longitudinal calculations cheat sheet*
   - *These analytical calculations serve to benchmark the tracking simulations.*
   - *You can also compare with the accelerating cases.*


In [ ]:
# Back to the injection energy: same cell as in Exercise 1, to overwrite the values at 14 GeV
# To compare lower/higher energies, change Ekin here (e.g. 1.3e12) and run the next cell again

Ekin = 26e9  # eV
voltage = 15e6  # V

charge = 1  # in units of e
circumference = 6911.5  # m
harmonic = 4620
gamma_t = 18

# Energy and momentum, in eV and eV/c
E0 = m_p * c**2 / e
energy = Ekin + E0
momentum = np.sqrt(energy**2 - E0**2)
beta = momentum / energy
gamma = energy / E0

# Revolution and rf frequencies
t_rev = circumference / (beta * c)
f_rev = 1 / t_rev
f_rf = harmonic * f_rev
t_rf = 1 / f_rf

# Momentum compaction and phase slippage factor
alpha_c = 1 / gamma_t**2
eta = alpha_c - 1 / gamma**2

print("Kinetic energy: " + str(Ekin / 1e9) + " GeV")
print("Beta: " + str(beta))
print("Gamma: " + str(gamma))
print("Revolution period: " + str(t_rev * 1e6) + " mus")
print("RF frequency: " + str(f_rf / 1e6) + " MHz")
print("RF period: " + str(t_rf * 1e9) + " ns")
print("Momentum compaction factor: " + str(alpha_c))
print("Phase slippage factor: " + str(eta))

omega_rev = 2 * np.pi * f_rev
omega_rf = harmonic * omega_rev

In [ ]:
# Analytical values, with the formulas of the cheat sheet
# - stable phase with the acceleration U0 of Exercise 4
# - height of the stationary bucket, and its reduction factor with acceleration
# - area of the stationary bucket, and its reduction factor with acceleration
# - synchrotron frequency and tune at the centre of the stationary bucket (phi_s = pi above transition)
# - print them and compare with your tracking plots

# --- Solution ---
phi_s = np.arcsin(U0 / (charge * voltage))

stationaryBucketHeight = beta * np.sqrt(
    2 * charge * voltage * energy / (np.pi * harmonic * np.abs(eta))
)
print("Bucket height: " + str(stationaryBucketHeight / 1e6) + " MeV")

bucketHeightRed = np.abs(np.cos(phi_s) - (np.pi - 2 * phi_s) / 2 * np.sin(phi_s)) ** (1 / 2)
print("Accelerating bucket height: " + str(stationaryBucketHeight / 1e6 * bucketHeightRed) + " MeV")

stationaryBucketArea = (
    16 * beta / omega_rf * np.sqrt(charge * voltage * energy / (2 * np.pi * harmonic * np.abs(eta)))
)
print("Bucket area: " + str(stationaryBucketArea) + " eVs")

bucketAreaRed = (1 - np.sin(phi_s)) / (1 + np.sin(phi_s))
print("Accelerating bucket area: " + str(stationaryBucketArea * bucketAreaRed) + " eVs")

synchrotronFrequency = np.sqrt(
    -(omega_rev**2)
    * harmonic
    * eta
    * charge
    * voltage
    * np.cos(np.pi)
    / (2 * np.pi * beta**2 * energy)
) / (2 * np.pi)
synchrotronTune = synchrotronFrequency / f_rev
print("Synchrotron frequency: " + str(synchrotronFrequency) + " Hz")
print("Synchrotron period: " + str(1 / synchrotronFrequency * 1e3) + " ms")
print("Synchrotron period: " + str(1 / synchrotronTune) + " turns")

# Answer: comparison with the tracking, the bucket height is the top of the separatrix of Exercise 3
# (at 15 MV), the accelerating values match the smaller separatrix of Exercise 4, and the
# synchrotron period in turns can be counted on the phase vs. turn plot of Exercise 2

## Exercise 7 (optional): Non-linear synchrotron frequency distribution, comparison with analytical formula

*Lecture slides: Tecker 39, 49, 51, 83*

1. Track a few tens of particles for a few synchrotron periods to analyze the frequency of synchrotron oscillation.
   - *You can start with the same script as in Exercise 3 to start with few particles.*
   - *The function `oscillation_spectrum` returns the spectrum of phase or energy oscillations for a given particle.*
   - *The function `synchrotron_tune` returns the tune of a given particle, based on the maximum of the oscillation spectrum obtained with FFT.*
2. Plot the synchrotron frequency versus phase or energy offset. This illustrates the synchrotron frequency distribution.
   - *Beware of particles extremely close to the center of the bucket or exactly on the separatrix*
3. Compare with the expected dependence of the non-linear synchrotron tune from the cheat sheet. *(Tecker 51, 83)*


In [ ]:
# Many particles at different amplitudes

n_particles = 100
n_turns = 1000

# - space the particles in phase between the centre of the bucket and the separatrix
#   (avoid the exact centre and the exact separatrix), at zero energy offset, and track them

# --- Solution ---
particlePhase = np.linspace(0.01 * np.pi, 0.99 * np.pi, n_particles)
particleEnergy = np.zeros(n_particles)

particlePhaseArray, particleEnergyArray = track(
    particlePhase, particleEnergy, n_turns, harmonic, eta, beta, energy, charge, voltage
)

In [ ]:
# Spectrum of the phase oscillation of a few particles

from support_functions import oscillation_spectrum

# - particlePhaseArray[:, 10] is the phase of particle number 10 at all turns
# - oscillation_spectrum returns the frequencies (in units of the revolution frequency, i.e. the tune)
#   and the amplitudes of the spectrum; its fft_zero_padding argument (e.g. 10000) refines the spectrum
# - plot the spectrum of a few particles, from the centre to the edge of the bucket

# --- Solution ---
tune_10, spectrum_10 = oscillation_spectrum(particlePhaseArray[:, 10], fft_zero_padding=10000)
tune_40, spectrum_40 = oscillation_spectrum(particlePhaseArray[:, 40], fft_zero_padding=10000)
tune_80, spectrum_80 = oscillation_spectrum(particlePhaseArray[:, 80], fft_zero_padding=10000)

plt.figure()
plt.plot(tune_10, spectrum_10)
plt.plot(tune_40, spectrum_40)
plt.plot(tune_80, spectrum_80)
plt.xlim(0, 0.05)
plt.xlabel("Synchrotron tune")
plt.ylabel("Amplitude [norm.]")

In [ ]:
# Synchrotron tune vs. amplitude of oscillation

from support_functions import synchrotron_tune

# - synchrotron_tune returns the amplitude of the phase oscillation and the tune of one particle
# - loop over the particles to fill two arrays, and plot the tune vs. the amplitude
# - compare with the small amplitude tune and its amplitude dependence, from the cheat sheet

# --- Solution ---
# Small amplitude synchrotron tune (phi_s = pi above transition, so -cos(phi_s) = 1)
synchrotronTune0 = np.sqrt(harmonic * eta * charge * voltage / (2 * np.pi * beta**2 * energy))

# Amplitude and tune of each tracked particle, from the peak of its oscillation spectrum
oscillationAmplitude = np.zeros(n_particles)
synchrotronTuneTracked = np.zeros(n_particles)
for idx_part in range(n_particles):
    oscillationAmplitude[idx_part], synchrotronTuneTracked[idx_part] = synchrotron_tune(
        particlePhaseArray[:, idx_part], fft_zero_padding=10000
    )

# Approximation of the cheat sheet for the tune vs. maximum phase amplitude
amplitudeArray = np.linspace(0, np.pi, 100, endpoint=False)
synchrotronTuneApprox = synchrotronTune0 * (1 - amplitudeArray**2 / 16)

plt.figure()
plt.plot(amplitudeArray, synchrotronTuneApprox, label="Approximation")
plt.plot(oscillationAmplitude, synchrotronTuneTracked, ".", label="Tracking")
plt.xlim(0, np.pi)
plt.xlabel(r"Maximum phase amplitude $\phi$ [rad]")
plt.ylabel("Synchrotron tune")
plt.legend()

# Answer: the synchrotron tune decreases with the oscillation amplitude, down to zero on the separatrix,
# a bunch filling the bucket has a whole distribution of synchrotron frequencies